# Jakemore: JAX vs. PyTorch Microbenchmarks & Manual Backprop

This notebook benchmarks a 2-layer character-level MLP with BatchNorm (`4,137` parameters, `batch_size = 32`, `block_size = 3`) across **PyTorch** (`eager` vs. `torch.compile`) and **JAX** (`eager` vs. `jax.jit`):

1. **Setup & Dataset Construction** (`X_train_*`, `Y_train_*`, and fixed benchmark mini-batches)
2. **Model Parameter Initialization** (`init_params_jax`, `init_params_torch`)
3. **Forward Pass Implementations** (`forward_jax`, `forward_torch`)
4. **Microbenchmark 1: Forward Pass Only** (`torch.inference_mode` vs. `jax.block_until_ready`)
5. **Manual Backward Pass Implementations & Gradient Checks** (`backward_manual_torch`, `backward_manual_jax`)
6. **Microbenchmark 2: Backward Pass Only** (PyTorch eager / compiled vs. JAX eager / JIT)
7. **Microbenchmark 3: Combined Forward + Backward Pass** (Autograd / `value_and_grad`)
8. **Appendix: Analytical Backprop Exercises 2–4 & JIT Training Loop**

## 1. Imports & Dataset Preparation

In [115]:
import random
import jax
import jax.numpy as jnp
import optax
import torch
import torch.nn.functional as F

print("JAX version:    ", jax.__version__, "| devices:", jax.devices())
print("PyTorch version:", torch.__version__)

JAX version:     0.11.1 | devices: [CpuDevice(id=0)]
PyTorch version: 2.11.0+cpu


In [116]:
# Download Karpathy's names.txt dataset
!wget -q -N https://raw.githubusercontent.com/karpathy/makemore/master/names.txt

In [117]:
# Load vocabulary and build character-to-index mappings
words = open("names.txt", "r").read().splitlines()
chars = sorted(list(set("".join(words))))
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi["."] = 0
itos = {i: s for s, i in stoi.items()}
vocab_size = len(itos)
print(f"Loaded {len(words):,} words | Vocabulary size: {vocab_size}")

Loaded 32,033 words | Vocabulary size: 27


In [118]:
# Build n-gram splits (80% train, 10% dev, 10% test)
block_size = 3  # context length: number of characters used to predict the next character


def build_dataset(word_list):
    X, Y = [], []
    for w in word_list:
        context = [0] * block_size
        for ch in w + ".":
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    return X, Y


random.seed(42)
random.shuffle(words)
n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))

X_train_list, Y_train_list = build_dataset(words[:n1])
X_dev_list, Y_dev_list = build_dataset(words[n1:n2])
X_test_list, Y_test_list = build_dataset(words[n2:])

# Separate PyTorch tensors and JAX arrays so neither framework overwrites the other
X_train_torch, Y_train_torch = torch.tensor(X_train_list, dtype=torch.long), torch.tensor(Y_train_list, dtype=torch.long)
X_dev_torch, Y_dev_torch = torch.tensor(X_dev_list, dtype=torch.long), torch.tensor(Y_dev_list, dtype=torch.long)
X_test_torch, Y_test_torch = torch.tensor(X_test_list, dtype=torch.long), torch.tensor(Y_test_list, dtype=torch.long)

X_train_jax, Y_train_jax = jnp.array(X_train_list, dtype=jnp.int32), jnp.array(Y_train_list, dtype=jnp.int32)
X_dev_jax, Y_dev_jax = jnp.array(X_dev_list, dtype=jnp.int32), jnp.array(Y_dev_list, dtype=jnp.int32)
X_test_jax, Y_test_jax = jnp.array(X_test_list, dtype=jnp.int32), jnp.array(Y_test_list, dtype=jnp.int32)

print("Train shape:", X_train_jax.shape, Y_train_jax.shape)
print("Dev shape:  ", X_dev_jax.shape, Y_dev_jax.shape)
print("Test shape: ", X_test_jax.shape, Y_test_jax.shape)

Train shape: (182625, 3) (182625,)
Dev shape:   (22655, 3) (22655,)
Test shape:  (22866, 3) (22866,)


## 2. Model Parameter Initialization & Benchmark Mini-Batches

In [119]:
def init_params_jax(key, n_embd, n_hidden):
    k1, k2, k3, k4, k5, k6, k7 = jax.random.split(key, 7)
    C = jax.random.normal(k1, (vocab_size, n_embd))
    W1 = jax.random.normal(k2, (n_embd * block_size, n_hidden)) * (5 / 3) / ((n_embd * block_size) ** 0.5)
    b1 = jax.random.normal(k3, (n_hidden,)) * 0.1
    W2 = jax.random.normal(k4, (n_hidden, vocab_size)) * 0.1
    b2 = jax.random.normal(k5, (vocab_size,)) * 0.1
    bngain = jax.random.normal(k6, (1, n_hidden)) * 0.1 + 1.0
    bnbias = jax.random.normal(k7, (1, n_hidden)) * 0.1
    return {
        "C": C,
        "W1": W1,
        "b1": b1,
        "W2": W2,
        "b2": b2,
        "bngain": bngain,
        "bnbias": bnbias,
    }


def init_params_torch(rng, n_embd, n_hidden):
    C = torch.randn((vocab_size, n_embd), generator=rng)
    W1 = torch.randn((n_embd * block_size, n_hidden), generator=rng) * (5 / 3) / ((n_embd * block_size) ** 0.5)
    b1 = torch.randn(n_hidden, generator=rng) * 0.1
    W2 = torch.randn((n_hidden, vocab_size), generator=rng) * 0.1
    b2 = torch.randn(vocab_size, generator=rng) * 0.1
    bngain = torch.randn((1, n_hidden), generator=rng) * 0.1 + 1.0
    bnbias = torch.randn((1, n_hidden), generator=rng) * 0.1
    parameters = {
        "C": C,
        "W1": W1,
        "b1": b1,
        "W2": W2,
        "b2": b2,
        "bngain": bngain,
        "bnbias": bnbias,
    }
    for param in parameters.values():
        param.requires_grad_(True)
    return parameters

In [120]:
n_embd = 10
n_hidden = 64
batch_size = 32

jax_key = jax.random.key(2147483647)
torch_rng = torch.Generator().manual_seed(2147483647)

jax_key, init_key_jax, batch_key_jax = jax.random.split(jax_key, 3)
jax_params = init_params_jax(init_key_jax, n_embd, n_hidden)
torch_params = init_params_torch(torch_rng, n_embd, n_hidden)

# Sample dedicated mini-batches for PyTorch and JAX
batch_indices_torch = torch.randint(0, X_train_torch.shape[0], (batch_size,), generator=torch_rng)
X_batch_torch, Y_batch_torch = X_train_torch[batch_indices_torch], Y_train_torch[batch_indices_torch]

batch_indices_jax = jax.random.randint(batch_key_jax, (batch_size,), 0, X_train_jax.shape[0])
X_batch_jax, Y_batch_jax = X_train_jax[batch_indices_jax], Y_train_jax[batch_indices_jax]

print("JAX parameter count:    ", sum(p.size for p in jax_params.values()))
print("PyTorch parameter count:", sum(p.nelement() for p in torch_params.values()))

JAX parameter count:     4137
PyTorch parameter count: 4137


## 3. Forward Pass Implementations (`forward_jax` & `forward_torch`)

In [121]:
def forward_jax(params, X_batch, Y_batch, return_intermediates=False, use_fast_cross_entropy=True):
    bsz = X_batch.shape[0]
    intermediates = {}

    emb = params["C"][X_batch]
    embcat = emb.reshape(bsz, -1)
    hprebn = embcat @ params["W1"] + params["b1"]

    # BatchNorm1d with Bessel's correction (ddof=1)
    bnmean = hprebn.mean(0, keepdims=True)
    bnvar = hprebn.var(0, keepdims=True, ddof=1)
    bnvar_inv = (bnvar + 1e-5) ** -0.5
    bnraw = (hprebn - bnmean) * bnvar_inv
    hpreact = params["bngain"] * bnraw + params["bnbias"]

    h = jax.nn.tanh(hpreact)
    logits = h @ params["W2"] + params["b2"]

    if use_fast_cross_entropy:
        loss = optax.softmax_cross_entropy_with_integer_labels(logits, Y_batch).mean()
    else:
        logit_maxes = logits.max(1, keepdims=True)
        norm_logits = logits - logit_maxes
        counts = jnp.exp(norm_logits)
        counts_sum = counts.sum(1, keepdims=True)
        counts_sum_inv = 1.0 / counts_sum
        probs = counts * counts_sum_inv
        logprobs = jnp.log(probs)
        loss = -logprobs[jnp.arange(bsz), Y_batch].mean()

        intermediates.update({
            "logit_maxes": logit_maxes,
            "norm_logits": norm_logits,
            "counts": counts,
            "counts_sum": counts_sum,
            "counts_sum_inv": counts_sum_inv,
            "probs": probs,
            "logprobs": logprobs,
        })

    if return_intermediates:
        intermediates.update({
            "emb": emb,
            "embcat": embcat,
            "hprebn": hprebn,
            "bnmean": bnmean,
            "bnvar": bnvar,
            "bnvar_inv": bnvar_inv,
            "bnraw": bnraw,
            "hpreact": hpreact,
            "h": h,
            "logits": logits,
            "loss": loss,
        })
        return loss, intermediates
    return loss


def forward_torch(params, X_batch, Y_batch, return_intermediates=False, use_fast_cross_entropy=True):
    bsz = X_batch.shape[0]
    intermediates = {}

    emb = params["C"][X_batch]
    embcat = emb.reshape(bsz, -1)
    hprebn = embcat @ params["W1"] + params["b1"]

    # BatchNorm1d with Bessel's correction (correction=1)
    bnmean = hprebn.mean(0, keepdim=True)
    bnvar = hprebn.var(0, keepdim=True, correction=1)
    bnvar_inv = (bnvar + 1e-5) ** -0.5
    bnraw = (hprebn - bnmean) * bnvar_inv
    hpreact = params["bngain"] * bnraw + params["bnbias"]

    h = torch.tanh(hpreact)
    logits = h @ params["W2"] + params["b2"]

    if use_fast_cross_entropy:
        loss = F.cross_entropy(logits, Y_batch)
    else:
        logit_maxes = logits.max(1, keepdim=True).values
        norm_logits = logits - logit_maxes
        counts = norm_logits.exp()
        counts_sum = counts.sum(1, keepdim=True)
        counts_sum_inv = 1.0 / counts_sum
        probs = counts * counts_sum_inv
        logprobs = probs.log()
        loss = -logprobs[torch.arange(bsz, device=logits.device), Y_batch].mean()

        intermediates.update({
            "logit_maxes": logit_maxes,
            "norm_logits": norm_logits,
            "counts": counts,
            "counts_sum": counts_sum,
            "counts_sum_inv": counts_sum_inv,
            "probs": probs,
            "logprobs": logprobs,
        })

    if return_intermediates:
        intermediates.update({
            "emb": emb,
            "embcat": embcat,
            "hprebn": hprebn,
            "bnmean": bnmean,
            "bnvar": bnvar,
            "bnvar_inv": bnvar_inv,
            "bnraw": bnraw,
            "hpreact": hpreact,
            "h": h,
            "logits": logits,
            "loss": loss,
        })
        return loss, intermediates
    return loss

## 4. Microbenchmark 1: Forward Pass Only

* **PyTorch** runs inside `torch.inference_mode()` so autograd graph construction does not penalize forward-only timing.
* **JAX** calls `jax.block_until_ready(...)` so asynchronous dispatch waits for device execution to finish.

In [122]:
# Compile and warm up Forward-Only functions before running %%timeit cells
forward_torch_compiled = torch.compile(forward_torch)
with torch.inference_mode():
    _ = forward_torch(torch_params, X_batch_torch, Y_batch_torch)
    _ = forward_torch_compiled(torch_params, X_batch_torch, Y_batch_torch)

forward_jax_jit = jax.jit(
    forward_jax,
    static_argnames=("return_intermediates", "use_fast_cross_entropy"),
)
jax.block_until_ready(forward_jax(jax_params, X_batch_jax, Y_batch_jax))
jax.block_until_ready(forward_jax_jit(jax_params, X_batch_jax, Y_batch_jax))

Array(3.3611188, dtype=float32)

In [123]:
%%timeit
# 4.1 PyTorch Forward-Only (Eager)
with torch.inference_mode():
    _ = forward_torch(torch_params, X_batch_torch, Y_batch_torch)

456 µs ± 110 µs per loop (mean ± std. dev. of 7 runs, 1000 loops each)


In [124]:
%%timeit
# 4.2 PyTorch Forward-Only (torch.compile)
with torch.inference_mode():
    _ = forward_torch_compiled(torch_params, X_batch_torch, Y_batch_torch)

383 µs ± 135 µs per loop (mean ± std. dev. of 7 runs, 1000 loops each)


In [125]:
%%timeit
# 4.3 JAX Forward-Only (Eager)
jax.block_until_ready(forward_jax(jax_params, X_batch_jax, Y_batch_jax))

4.27 ms ± 1.06 ms per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [126]:
%%timeit
# 4.4 JAX Forward-Only (jax.jit)
jax.block_until_ready(forward_jax_jit(jax_params, X_batch_jax, Y_batch_jax))

80.5 µs ± 2.8 µs per loop (mean ± std. dev. of 7 runs, 10000 loops each)


## 5. Manual Backward Pass Implementations & Gradient Verification

Both `backward_manual_torch` and `backward_manual_jax` take `(intermediates, params, X_batch, Y_batch)` explicitly so `torch.compile` and `jax.jit` trace dynamic batch inputs rather than freezing global variables.

In [127]:
@torch.no_grad()
def backward_manual_torch(intermediates, params, X_batch, Y_batch):
    bsz = X_batch.shape[0]
    grad = {}

    logprobs = intermediates["logprobs"]
    dlogprobs = torch.zeros_like(logprobs)
    dlogprobs[torch.arange(bsz, device=logprobs.device), Y_batch] = -1.0 / bsz

    probs = intermediates["probs"]
    dprobs = (1.0 / probs) * dlogprobs

    counts_sum_inv = intermediates["counts_sum_inv"]
    counts = intermediates["counts"]
    dcounts_sum_inv = (counts * dprobs).sum(1, keepdim=True)
    dcounts = counts_sum_inv * dprobs

    counts_sum = intermediates["counts_sum"]
    dcounts_sum = -(counts_sum ** -2) * dcounts_sum_inv
    dcounts = dcounts + dcounts_sum

    norm_logits = intermediates["norm_logits"]
    dnorm_logits = norm_logits.exp() * dcounts

    dlogit_maxes = -dnorm_logits.sum(1, keepdim=True)
    logits = intermediates["logits"]
    max_indices = logits.argmax(dim=1)
    dlogits = dnorm_logits + F.one_hot(max_indices, num_classes=logits.shape[1]).to(logits.dtype) * dlogit_maxes

    h = intermediates["h"]
    dh = dlogits @ params["W2"].T
    grad["W2"] = h.T @ dlogits
    grad["b2"] = dlogits.sum(0)

    dhpreact = (1.0 - h ** 2) * dh

    bnraw = intermediates["bnraw"]
    grad["bngain"] = (bnraw * dhpreact).sum(0, keepdim=True)
    grad["bnbias"] = dhpreact.sum(0, keepdim=True)

    dbnraw = params["bngain"] * dhpreact
    hprebn = intermediates["hprebn"]
    bnmean = intermediates["bnmean"]
    dbnvar_inv = ((hprebn - bnmean) * dbnraw).sum(0, keepdim=True)

    bnvar = intermediates["bnvar"]
    dbnvar = -0.5 * ((bnvar + 1e-5) ** -1.5) * dbnvar_inv

    bnvar_inv = intermediates["bnvar_inv"]
    dbnmean = (-bnvar_inv * dbnraw).sum(0, keepdim=True)

    dhprebn = bnvar_inv * dbnraw
    dhprebn = dhprebn + (2.0 / (bsz - 1)) * (hprebn - bnmean) * dbnvar
    dhprebn = dhprebn + (1.0 / bsz) * dbnmean

    dembcat = dhprebn @ params["W1"].T
    grad["W1"] = intermediates["embcat"].T @ dhprebn
    grad["b1"] = dhprebn.sum(0)

    emb = intermediates["emb"]
    demb = dembcat.reshape(emb.shape)
    one_hot_X = F.one_hot(X_batch, num_classes=vocab_size).to(demb.dtype)
    grad["C"] = torch.bmm(one_hot_X.transpose(1, 2), demb).sum(0)

    return grad

In [128]:
# Verify backward_manual_torch against PyTorch autograd
for p in torch_params.values():
    p.grad = None

loss_torch_ref, intermediates_torch_ag = forward_torch(
    torch_params,
    X_batch_torch,
    Y_batch_torch,
    return_intermediates=True,
    use_fast_cross_entropy=False,
)
loss_torch_ref.backward()
grads_autograd_torch = {k: p.grad.detach().clone() for k, p in torch_params.items()}

# Detached intermediates for standalone backward-only benchmarking
intermediates_torch = {k: v.detach() for k, v in intermediates_torch_ag.items()}
grads_manual_torch = backward_manual_torch(intermediates_torch, torch_params, X_batch_torch, Y_batch_torch)

for k in torch_params:
    max_diff = (grads_autograd_torch[k] - grads_manual_torch[k]).abs().max().item()
    is_close = torch.allclose(grads_autograd_torch[k], grads_manual_torch[k], atol=1e-6)
    print(f"PyTorch grad[{k:6s}] | allclose={str(is_close):5s} | max_diff={max_diff:.3e}")
    assert is_close, f"Gradient mismatch on {k}: {max_diff}"

for p in torch_params.values():
    p.grad = None

PyTorch grad[C     ] | allclose=True  | max_diff=1.327e-08
PyTorch grad[W1    ] | allclose=True  | max_diff=1.118e-08
PyTorch grad[b1    ] | allclose=True  | max_diff=5.588e-09
PyTorch grad[W2    ] | allclose=True  | max_diff=2.421e-08
PyTorch grad[b2    ] | allclose=True  | max_diff=2.980e-08
PyTorch grad[bngain] | allclose=True  | max_diff=4.889e-09
PyTorch grad[bnbias] | allclose=True  | max_diff=9.313e-09


In [131]:
def backward_manual_jax(intermediates, params, X_batch, Y_batch):
    bsz = X_batch.shape[0]
    grad = {}

    logprobs = intermediates["logprobs"]
    dlogprobs = jnp.zeros_like(logprobs).at[jnp.arange(bsz), Y_batch].set(-1.0 / bsz)

    probs = intermediates["probs"]
    dprobs = (1.0 / probs) * dlogprobs

    counts_sum_inv = intermediates["counts_sum_inv"]
    counts = intermediates["counts"]
    dcounts_sum_inv = (counts * dprobs).sum(1, keepdims=True)
    dcounts = counts_sum_inv * dprobs

    counts_sum = intermediates["counts_sum"]
    dcounts_sum = -(counts_sum ** -2) * dcounts_sum_inv
    dcounts = dcounts + dcounts_sum

    norm_logits = intermediates["norm_logits"]
    dnorm_logits = jnp.exp(norm_logits) * dcounts

    dlogit_maxes = -dnorm_logits.sum(1, keepdims=True)
    logits = intermediates["logits"]
    dlogits = dnorm_logits + jax.nn.one_hot(logits.argmax(1), num_classes=logits.shape[1]) * dlogit_maxes

    h = intermediates["h"]
    dh = dlogits @ params["W2"].T
    grad["W2"] = h.T @ dlogits
    grad["b2"] = dlogits.sum(0)

    dhpreact = (1.0 - h ** 2) * dh

    bnraw = intermediates["bnraw"]
    grad["bngain"] = (bnraw * dhpreact).sum(0, keepdims=True)
    grad["bnbias"] = dhpreact.sum(0, keepdims=True)

    dbnraw = params["bngain"] * dhpreact
    hprebn = intermediates["hprebn"]
    bnmean = intermediates["bnmean"]
    dbnvar_inv = ((hprebn - bnmean) * dbnraw).sum(0, keepdims=True)

    bnvar = intermediates["bnvar"]
    dbnvar = -0.5 * ((bnvar + 1e-5) ** -1.5) * dbnvar_inv

    bnvar_inv = intermediates["bnvar_inv"]
    dbnmean = (-bnvar_inv * dbnraw).sum(0, keepdims=True)

    dhprebn = bnvar_inv * dbnraw
    dhprebn = dhprebn + (2.0 / (bsz - 1)) * (hprebn - bnmean) * dbnvar
    dhprebn = dhprebn + (1.0 / bsz) * dbnmean

    dembcat = dhprebn @ params["W1"].T
    grad["W1"] = intermediates["embcat"].T @ dhprebn
    grad["b1"] = dhprebn.sum(0)

    emb = intermediates["emb"]
    demb = dembcat.reshape(emb.shape)
    one_hot_X = jax.nn.one_hot(X_batch, num_classes=vocab_size).transpose((0, 2, 1))
    grad["C"] = jax.vmap(jnp.matmul, in_axes=(0, 0))(one_hot_X, demb).sum(0)

    return grad

In [132]:
# Verify backward_manual_jax against jax.value_and_grad using jax.tree.leaves(...)
(loss_jax_ref, intermediates_jax), grads_autodiff_jax = jax.value_and_grad(
    forward_jax,
    has_aux=True,
)(
    jax_params,
    X_batch_jax,
    Y_batch_jax,
    return_intermediates=True,
    use_fast_cross_entropy=False,
)

grads_manual_jax = backward_manual_jax(intermediates_jax, jax_params, X_batch_jax, Y_batch_jax)

for k in jax_params:
    max_diff = float(jnp.max(jnp.abs(grads_autodiff_jax[k] - grads_manual_jax[k])))
    is_close = bool(jnp.allclose(grads_autodiff_jax[k], grads_manual_jax[k], atol=1e-6))
    print(f"JAX grad[{k:6s}]     | allclose={str(is_close):5s} | max_diff={max_diff:.3e}")

assert all(
    jax.tree.leaves(
        jax.tree.map(lambda a, b: jnp.allclose(a, b, atol=1e-6), grads_autodiff_jax, grads_manual_jax)
    )
)

JAX grad[C     ]     | allclose=True  | max_diff=1.118e-08
JAX grad[W1    ]     | allclose=True  | max_diff=4.657e-09
JAX grad[b1    ]     | allclose=True  | max_diff=3.900e-09
JAX grad[W2    ]     | allclose=True  | max_diff=7.451e-09
JAX grad[b2    ]     | allclose=True  | max_diff=0.000e+00
JAX grad[bngain]     | allclose=True  | max_diff=2.561e-09
JAX grad[bnbias]     | allclose=True  | max_diff=5.588e-09


## 6. Microbenchmark 2: Backward Pass Only (Manual Backprop)

Benchmarks the standalone backward pass (`backward_manual_torch` vs. `backward_manual_jax`) given pre-computed forward intermediates.

In [133]:
# Compile and warm up Backward-Only functions before running %%timeit cells
backward_manual_torch_compiled = torch.compile(backward_manual_torch)
_ = backward_manual_torch(intermediates_torch, torch_params, X_batch_torch, Y_batch_torch)
_ = backward_manual_torch_compiled(intermediates_torch, torch_params, X_batch_torch, Y_batch_torch)

backward_manual_jax_jit = jax.jit(backward_manual_jax)
jax.block_until_ready(backward_manual_jax(intermediates_jax, jax_params, X_batch_jax, Y_batch_jax))
jax.block_until_ready(backward_manual_jax_jit(intermediates_jax, jax_params, X_batch_jax, Y_batch_jax))

{'C': Array([[ 4.54877168e-02, -2.08532214e-02,  1.17238462e-02,
          8.58275034e-03,  7.57238269e-03, -2.47726273e-02,
          1.61102861e-02, -1.01909963e-02,  3.15292478e-02,
         -2.70967521e-02],
        [-1.67445987e-02, -7.82433432e-04, -1.85415093e-02,
          2.45350692e-03, -2.10739439e-03,  1.69901066e-02,
          2.21185070e-02, -6.57393364e-03, -6.91891136e-03,
          6.17046747e-03],
        [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
          0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
          0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
          0.00000000e+00],
        [ 1.60683889e-03,  4.91682859e-03, -2.36864784e-03,
         -1.55461312e-03,  1.55569997e-03,  8.01968155e-04,
         -1.65492052e-03, -1.13562368e-04,  7.77358888e-03,
          6.07736874e-03],
        [ 3.20897135e-03,  4.41150833e-03, -2.25526723e-03,
          3.59345833e-03, -7.62464711e-03,  2.06241253e-04,
         -3.15164821e-03,  6.24994980e-03, -6.7

In [134]:
%%timeit
# 6.1 PyTorch Backward-Only (Eager)
_ = backward_manual_torch(intermediates_torch, torch_params, X_batch_torch, Y_batch_torch)

607 µs ± 99.3 µs per loop (mean ± std. dev. of 7 runs, 1000 loops each)


In [135]:
%%timeit
# 6.2 PyTorch Backward-Only (torch.compile)
_ = backward_manual_torch_compiled(intermediates_torch, torch_params, X_batch_torch, Y_batch_torch)

183 µs ± 24.8 µs per loop (mean ± std. dev. of 7 runs, 10000 loops each)


In [136]:
%%timeit
# 6.3 JAX Backward-Only (Eager)
jax.block_until_ready(backward_manual_jax(intermediates_jax, jax_params, X_batch_jax, Y_batch_jax))

6.98 ms ± 65.4 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [137]:
%%timeit
# 6.4 JAX Backward-Only (jax.jit)
jax.block_until_ready(backward_manual_jax_jit(intermediates_jax, jax_params, X_batch_jax, Y_batch_jax))

295 µs ± 59.6 µs per loop (mean ± std. dev. of 7 runs, 1000 loops each)


## 7. Microbenchmark 3: Combined Forward + Backward Pass (Autodiff)

In [138]:
# Define, compile, and warm up combined Forward + Backward functions
def forward_and_backward_torch(params, X_batch, Y_batch):
    loss = forward_torch(params, X_batch, Y_batch)
    loss.backward()
    for p in params.values():
        p.grad = None
    return loss


forward_and_backward_torch_compiled = torch.compile(forward_and_backward_torch)
_ = forward_and_backward_torch(torch_params, X_batch_torch, Y_batch_torch)
_ = forward_and_backward_torch_compiled(torch_params, X_batch_torch, Y_batch_torch)

forward_and_backward_jax = jax.value_and_grad(forward_jax)
forward_and_backward_jax_jit = jax.jit(
    forward_and_backward_jax,
    static_argnames=("return_intermediates", "use_fast_cross_entropy"),
)
jax.block_until_ready(forward_and_backward_jax(jax_params, X_batch_jax, Y_batch_jax))
jax.block_until_ready(forward_and_backward_jax_jit(jax_params, X_batch_jax, Y_batch_jax))

(Array(3.3611188, dtype=float32),
 {'C': Array([[ 4.5487732e-02, -2.0853210e-02,  1.1723847e-02,  8.5827447e-03,
           7.5723771e-03, -2.4772627e-02,  1.6110290e-02, -1.0191003e-02,
           3.1529255e-02, -2.7096750e-02],
         [-1.6744601e-02, -7.8243902e-04, -1.8541509e-02,  2.4535058e-03,
          -2.1073967e-03,  1.6990107e-02,  2.2118505e-02, -6.5739355e-03,
          -6.9189090e-03,  6.1704670e-03],
         [ 0.0000000e+00,  0.0000000e+00,  0.0000000e+00,  0.0000000e+00,
           0.0000000e+00,  0.0000000e+00,  0.0000000e+00,  0.0000000e+00,
           0.0000000e+00,  0.0000000e+00],
         [ 1.6068391e-03,  4.9168286e-03, -2.3686476e-03, -1.5546106e-03,
           1.5557000e-03,  8.0196728e-04, -1.6549212e-03, -1.1356320e-04,
           7.7735884e-03,  6.0773711e-03],
         [ 3.2089713e-03,  4.4115093e-03, -2.2552682e-03,  3.5934574e-03,
          -7.6246471e-03,  2.0624143e-04, -3.1516487e-03,  6.2499489e-03,
          -6.7165156e-04, -2.4566674e-03],
      

In [139]:
%%timeit
# 7.1 PyTorch Forward + Backward (Eager Autograd)
_ = forward_and_backward_torch(torch_params, X_batch_torch, Y_batch_torch)

718 µs ± 140 µs per loop (mean ± std. dev. of 7 runs, 1000 loops each)


In [140]:
%%timeit
# 7.2 PyTorch Forward + Backward (torch.compile Autograd)
_ = forward_and_backward_torch_compiled(torch_params, X_batch_torch, Y_batch_torch)

737 µs ± 111 µs per loop (mean ± std. dev. of 7 runs, 1000 loops each)


In [141]:
%%timeit
# 7.3 JAX Forward + Backward (Eager jax.value_and_grad)
jax.block_until_ready(forward_and_backward_jax(jax_params, X_batch_jax, Y_batch_jax))

33.1 ms ± 935 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [142]:
%%timeit
# 7.4 JAX Forward + Backward (jax.jit(jax.value_and_grad))
jax.block_until_ready(forward_and_backward_jax_jit(jax_params, X_batch_jax, Y_batch_jax))

310 µs ± 11.1 µs per loop (mean ± std. dev. of 7 runs, 1000 loops each)
